In [ ]:
import matplotlib.font_manager
fonts = set(f.name for f in matplotlib.font_manager.fontManager.ttflist)
sans_fonts = [f for f in fonts if 'Sans' in f or 'sans' in f]
print("可用的无衬线字体:", sorted(sans_fonts))

In [ ]:
import matplotlib
# matplotlib.use('Agg') # Uncomment if running on server without display
import os
import numpy as np
import matplotlib.pyplot as plt
import cartopy.io.shapereader as shpreader
from matplotlib.collections import LineCollection
from datetime import datetime, timedelta

# --- 1. Configure paths ---
wrf_base_path = 'your_path/wrf'
wrf1h_path = 'your_path/d03_d06_pm25'
les_base_path = 'your_path/moni_pm25'
dl_base_path = 'your_path/result'
shapefile_path = 'your_path/gadm41_CHN_1.shp'

# --- 2. Set parameters ---
time_list = ['2024030105', '2024030117']
i_range = range(1, 8)   # 7 tiles horizontally
j_range = range(1, 12)  # 11 tiles vertically

# Tile size (based on slicing: 100-20=80, 140-20=120)
tile_h, tile_w = 80, 120 
total_h = len(j_range) * tile_h
total_w = len(i_range) * tile_w

# --- Plot style settings ---
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 15,
    "mathtext.fontset": "dejavusans",
    "figure.titlesize": 10,
})
time_labels = ['(a)', '(b)', '(c)', '(d)']

# ==========================================
# Utility 1: Build lon/lat to pixel coordinate transformer
# ==========================================
def build_coordinate_transformer(grid_lon, grid_lat):
    """
    利用最小二乘法，建立 (lon, lat) 到大矩阵像素 (x, y) 的映射公式。
    解决 WRF 投影导致省界在这个正方形网格上对不准的问题。
    """
        # Construct pixel coordinate matrices X, Y for grid points
    H, W = grid_lon.shape
    X, Y = np.meshgrid(np.arange(W), np.arange(H))
    
        # Downsample (take every 10th point) to speed up fitting
    step = 10
    flat_lon = grid_lon[::step, ::step].flatten()
    flat_lat = grid_lat[::step, ::step].flatten()
    flat_x = X[::step, ::step].flatten()
    flat_y = Y[::step, ::step].flatten()
    
        # Remove invalid values
    valid = ~np.isnan(flat_lon)
    if np.sum(valid) < 10:
        return None
        
    flat_lon = flat_lon[valid]
    flat_lat = flat_lat[valid]
    flat_x = flat_x[valid]
    flat_y = flat_y[valid]

        # Quadratic polynomial fit: [1, lon, lat, lon^2, lon*lat, lat^2]
    A = np.column_stack([
        np.ones_like(flat_lon), flat_lon, flat_lat,
        flat_lon**2, flat_lon*flat_lat, flat_lat**2
    ])
    
        # Compute coefficients
    C_x, _, _, _ = np.linalg.lstsq(A, flat_x, rcond=None)
    C_y, _, _, _ = np.linalg.lstsq(A, flat_y, rcond=None)
    
        # Define internal transform function
    def transform(lon_arr, lat_arr):
        lon_arr = np.asarray(lon_arr)
        lat_arr = np.asarray(lat_arr)
        A_new = np.column_stack([
            np.ones_like(lon_arr), lon_arr, lat_arr,
            lon_arr**2, lon_arr*lat_arr, lat_arr**2
        ])
        return np.dot(A_new, C_x), np.dot(A_new, C_y)
    
    return transform

# ==========================================
# Utility 2: Draw Shapefile on Axes
# ==========================================
def add_shapefile_to_ax(ax, shape_path, transformer, bounds):
    if not os.path.exists(shape_path) or transformer is None:
        return

    reader = shpreader.Reader(shape_path)
    segments = []
    min_lon, max_lon, min_lat, max_lat = bounds
    
        # Iterate over provinces
    for geom in reader.geometries():
            # Simple filter: skip provinces outside range
        g_min_x, g_min_y, g_max_x, g_max_y = geom.bounds
        if (g_max_x < min_lon) or (g_min_x > max_lon) or \
           (g_max_y < min_lat) or (g_min_y > max_lat):
            continue

        if geom.geom_type == 'MultiPolygon':
            polys = list(geom.geoms)
        else:
            polys = [geom]
            
        for poly in polys:
                        # Get boundary lon/lat
            lons = np.array([p[0] for p in poly.exterior.coords])
            lats = np.array([p[1] for p in poly.exterior.coords])
            
                        # Core step: lon/lat to pixel coordinates
            xs, ys = transformer(lons, lats)
            
            segments.append(np.column_stack((xs, ys)))

        # Use LineCollection for efficient drawing
    lc = LineCollection(segments, colors='#444444', linewidths=0.6)
    ax.add_collection(lc)

# ==========================================
# 3. Main loop processing
# ==========================================
for idx, target_time in enumerate(time_list):
    print(f"正在处理时间: {target_time} ...")
    
    # --- Initialize large matrices (filled with NaN) ---
    grid_wrf = np.full((total_h, total_w), np.nan)
    grid_wrf1h = np.full((total_h, total_w), np.nan)
    grid_les = np.full((total_h, total_w), np.nan)
    grid_fno = np.full((total_h, total_w), np.nan)
    
    # Lat/lon matrices (for projection calculation)
    grid_lat = np.full((total_h, total_w), np.nan)
    grid_lon = np.full((total_h, total_w), np.nan)

    has_data = False

    # --- Loop through tiles and assemble into large matrices ---
    for i_idx, i in enumerate(i_range):
        for j_idx, j in enumerate(j_range):
            tile_name = f'i{i}_j{j}'
            
            # Calculate position in large image (origin='lower', j=1 at bottom)
            y_s = j_idx * tile_h
            y_e = (j_idx + 1) * tile_h
            x_s = i_idx * tile_w
            x_e = (i_idx + 1) * tile_w
            
            # 1. Read WRF
            f_wrf = os.path.join(wrf_base_path, tile_name, f'{target_time}.npy')
            if not os.path.exists(f_wrf):
                continue

            try:
                wrf_dict = np.load(f_wrf, allow_pickle=True).item()
                
                # Extract and crop [20:100, 20:140]
                wrf_full = wrf_dict['pm25']
                wrf_crop = wrf_full[20:100, 20:140]
                
                lat_full = wrf_dict['lat']
                lon_full = wrf_dict['lon']
                #print(lat_full)
                # Ensure lat/lon slice matches PM2.5
                lat_crop = lat_full[20:100, 20:140]
                lon_crop = lon_full[20:100, 20:140]
                
                # Handle case where lon/lat might be 1D
                if lon_crop.ndim == 1 and lat_crop.ndim == 1:
                    lon_crop, lat_crop = np.meshgrid(lon_crop, lat_crop)
                
                # Fill into large matrix
                grid_wrf[y_s:y_e, x_s:x_e] = wrf_crop
                grid_lat[y_s:y_e, x_s:x_e] = lat_crop
                grid_lon[y_s:y_e, x_s:x_e] = lon_crop
                
                has_data = True

                # 2. Read WRF+1h
                path_1h = os.path.join(wrf1h_path, tile_name, f'{target_time}.npy')
                if os.path.exists(path_1h):
                    try:
                        pm25_1h = np.load(path_1h, allow_pickle=True)
                        crop_1h = pm25_1h[10:90, 10:130] # Slice size must be 80x120
                        grid_wrf1h[y_s:y_e, x_s:x_e] = wrf_crop + crop_1h
                    except: pass

                # 3. Read LES
                path_les = os.path.join(les_base_path, tile_name, f'{target_time}.npy')
                if os.path.exists(path_les):
                    try:
                        pm25_les = np.load(path_les, allow_pickle=True)
                        crop_les = pm25_les[10:90, 10:130]
                        grid_les[y_s:y_e, x_s:x_e] = wrf_crop + crop_les
                    except: pass
                
                # 4. Read FNO (DL)
                path_dl = os.path.join(dl_base_path, tile_name, f'{target_time}.npy')
                if os.path.exists(path_dl):
                    try:
                        pm25_dl = np.load(path_dl, allow_pickle=True)
                        crop_dl = pm25_dl[10:90, 10:130]
                        grid_fno[y_s:y_e, x_s:x_e] = wrf_crop + crop_dl
                    except: pass

            except Exception as e:
                print(f"读取出错 {tile_name}: {e}")
                pass

    if not has_data:
        print(f"时间 {target_time} 没有有效数据，跳过。")
        continue

    # --- Prepare coordinate transformer ---
    print("正在计算省界投影变换...")
    # Get valid range for filtering Shapefile
    mask = ~np.isnan(grid_lon)
    if not np.any(mask):
        print("警告：经纬度数据全为空，无法绘制省界")
        map_bounds = [0, 0, 0, 0]
        transformer = None
    else:
        min_lon, max_lon = np.min(grid_lon[mask]), np.max(grid_lon[mask])
        min_lat, max_lat = np.min(grid_lat[mask]), np.max(grid_lat[mask])
        map_bounds = [min_lon - 0.5, max_lon + 0.5, min_lat - 0.5, max_lat + 0.5]
        transformer = build_coordinate_transformer(grid_lon, grid_lat)

    # --- Start plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
    vmin, vmax = 0, 60
    cmap = plt.get_cmap('jet')

    plot_data = [
        #(grid_wrf, 'Initial Fields'),
        (grid_wrf1h, 'WRF Assimilated Forecas'),
        (grid_les, 'LES Forecast'),
        (grid_fno, 'FNO Prediction')
    ]

    im_ref = None
    for ax, (data, title) in zip(axes, plot_data):
        # 1. Draw grid image (square, origin='lower')
        # interpolation='nearest' keeps pixel sharpness, avoids blur
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, 
                       origin='lower', aspect='equal', interpolation='nearest')
        
        # 2. Overlay transformed province boundaries
        if transformer:
            add_shapefile_to_ax(ax, shapefile_path, transformer, map_bounds)
        
        # 3. Set titles and remove axes
        ax.set_title(title, fontsize=15,  pad=8,fontweight='bold',)
        ax.axis('off') # Completely remove axes and borders, keep only image
        
        im_ref = im

    # Add panel labels (a)
    if idx==0:
        axes[0].text(0.02, 0.96, '(a)', transform=axes[0].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold', color='white')
        axes[1].text(0.02, 0.96, '(b)', transform=axes[1].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold', color='white')
        axes[2].text(0.02, 0.96, '(c)', transform=axes[2].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold', color='white')
    else:
        axes[0].text(0.02, 0.96, '(d)', transform=axes[0].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold', color='white')
        axes[1].text(0.02, 0.96, '(e)', transform=axes[1].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold', color='white')
        axes[2].text(0.02, 0.96, '(f)', transform=axes[2].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold', color='white')
                 #bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.9))
    #panel_label = time_labels[idx] if idx < len(time_labels) else "(a)"
    # Add label to top-left corner of first subplot
    #axes[0].text(0.02, 0.96, panel_label, transform=axes[0].transAxes,
    #             fontsize=14,  va='top', ha='left',fontweight='bold',)
                 #bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.9))

    # Add shared colorbar
    cbar = fig.colorbar(im_ref, ax=axes, shrink=0.75, aspect=25, pad=0.02)
    cbar.set_label('PM$_{2.5}$ ($\mu$g/m$^3$)', fontsize=15)
    cbar.ax.tick_params(labelsize=14)

    # Save image
    out_dir = 'your_path/code5FNO/Project/plot'
    os.makedirs(out_dir, exist_ok=True)
    out_file = os.path.join(out_dir,  f'JGR_PM25_{target_time}.png')
    
    # dpi=600 for high quality, bbox_inches='tight' auto-crops white margins
    plt.savefig(out_file, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"✅ 图片已保存: {out_file}")
    
    plt.show() # If on remote server, comment out this line
    plt.close(fig)

In [ ]:
# Cell 1: Interpolate model data to observation stations
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
from scipy.interpolate import RegularGridInterpolator
#99304	31.207100	121.577000	23.000000	6	8 your_path/stations_in_ij_2024030106.txt删除
# --- Configure paths ---
wrf_base_path = 'your_path/wrf'
wrf_1h_path = 'your_path/d03_d06_pm25'
les_base_path = 'your_path/moni_pm25'
dl_base_path = 'your_path/result'
obs_dir = 'your_path' 
output_dir = 'your_path/code5FNO/Project/plot'
os.makedirs(output_dir, exist_ok=True)

model_time_list = ['2024030105', '2024030117']

def get_interpolated_values(station_df, data_map_list, key_name):
    results = []
    for _, station in station_df.iterrows():
        lat, lon = station['lat'], station['lon']
        found_data = None
        for data in data_map_list:
            if (lat >= data['lat'].min() and lat <= data['lat'].max() and 
                lon >= data['lon'].min() and lon <= data['lon'].max()):
                found_data = data
                break
        if found_data is None:
            results.append(np.nan)
            continue
        try:
            lats = found_data['lat'][:, 0]  # (ny,)
            lons = found_data['lon'][0, :]  # (nx,)
            pm25 = found_data[key_name]
            interpolator = RegularGridInterpolator(
                (lats, lons), pm25, method='linear', bounds_error=False, fill_value=np.nan
            )
            val = interpolator([[lat, lon]])[0]
            results.append(val)
        except Exception as e:
            results.append(np.nan)
    return np.array(results)

# Store results for all times
all_results = {}

for model_time in model_time_list:
    dt_model = datetime.strptime(model_time, "%Y%m%d%H")
    obs_time = (dt_model + timedelta(hours=1)).strftime("%Y%m%d%H")
    obs_file = os.path.join(obs_dir, f'stations_in_ij_{obs_time}.txt')
    
    if not os.path.exists(obs_file):
        print(f"跳过 {model_time}: 观测文件不存在")
        continue
    
    try:
        df_obs = pd.read_csv(obs_file, sep='\t')
        if 'PM2.5' not in df_obs.columns:
            print(f"跳过 {model_time}: 无PM2.5列")
            continue
    except Exception as e:
        print(f"读取失败 {obs_file}: {e}")
        continue
    
    # 加载所有 tile
    combined_maps = []
    for i in range(1, 8):
        for j in range(1, 12):
            tile = f'i{i}_j{j}'
            f_wrf = os.path.join(wrf_base_path, tile, f'{model_time}.npy')
            if not os.path.exists(f_wrf): 
                continue
            try:
                wrf_dict = np.load(f_wrf, allow_pickle=True).item()
                w_crop = wrf_dict['pm25'][20:100, 20:140]
                
                f1_crop = np.load(os.path.join(wrf_1h_path, tile, f'{model_time}.npy'), allow_pickle=True)[10:90,10:130] if os.path.exists(os.path.join(wrf_1h_path, tile, f'{model_time}.npy')) else np.zeros_like(w_crop)
                les_crop = np.load(os.path.join(les_base_path, tile, f'{model_time}.npy'), allow_pickle=True)[10:90,10:130] if os.path.exists(os.path.join(les_base_path, tile, f'{model_time}.npy')) else np.zeros_like(w_crop)
                dl_crop = np.load(os.path.join(dl_base_path, tile, f'{model_time}.npy'), allow_pickle=True)[10:90,10:130] if os.path.exists(os.path.join(dl_base_path, tile, f'{model_time}.npy')) else np.zeros_like(w_crop)
                
                # Build complete field
                full_shape = wrf_dict['pm25'].shape
                def embed(crop):
                    m = np.full(full_shape, np.nan)
                    m[20:100, 20:140] = w_crop + crop
                    return m
                
                combined_maps.append({
                    'lat': wrf_dict['lat'],
                    'lon': wrf_dict['lon'],
                    'wrf1h': embed(f1_crop),
                    'les': embed(les_crop),
                    'dl': embed(dl_crop)
                })
            except Exception as e:
                continue
    
    if not combined_maps:
        print(f"跳过 {model_time}: 无模型数据")
        continue
    
    # 插值
    df_obs['wrf1h'] = get_interpolated_values(df_obs, combined_maps, 'wrf1h')
    df_obs['les'] = get_interpolated_values(df_obs, combined_maps, 'les')
    df_obs['dl'] = get_interpolated_values(df_obs, combined_maps, 'dl')
    
    # Save插值结果
    output_file = os.path.join(output_dir, f'interpolated_PM25_{model_time}.csv')
    df_obs.to_csv(output_file, index=False)
    print(f"✅ 已保存插值结果: {output_file}")
    
    all_results[model_time] = {
        'obs_time': obs_time,
        'df': df_obs
    }

print("Cell 1 完成：插值数据已保存。")

In [ ]:
# Cell 1: Station interpolation based on lat/lon (without i/j)
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
from scipy.interpolate import RegularGridInterpolator

# ------------------ Path configuration ------------------
wrf_base_path = 'your_path/wrf'
wrf_1h_path   = 'your_path/d03_d06_pm25'
les_base_path = 'your_path/moni_pm25'
obs_dir    = 'your_path'
os.makedirs(output_dir, exist_ok=True)

model_time_list = ['2024030105', '2024030117']

# ------------------ Interpolation function (pure lat/lon) ------------------
def interpolate_station_values(station_df, maps, varname):
    """
    station_df : DataFrame with lat, lon
    maps       : list of dict, each contains lat, lon, varname
    """
    results = []

    for _, row in station_df.iterrows():
        lat, lon = row['lat'], row['lon']
        val = np.nan

        for m in maps:
            lat2d, lon2d = m['lat'], m['lon']

            if (lat2d.min() <= lat <= lat2d.max() and
                lon2d.min() <= lon <= lon2d.max()):

                lats = lat2d[:, 0]
                lons = lon2d[0, :]

                interp = RegularGridInterpolator(
                    (lats, lons),
                    m[varname],
                    bounds_error=False,
                    fill_value=np.nan
                )

                val = interp([[lat, lon]])[0]
                break

        results.append(val)

    return np.array(results)
all_results = {}
# ------------------ Main loop ------------------
for model_time in model_time_list:

    print(f'\n=== 处理 {model_time} ===')

    dt_model = datetime.strptime(model_time, "%Y%m%d%H")
    obs_time = (dt_model + timedelta(hours=1)).strftime("%Y%m%d%H")
    obs_file = os.path.join(obs_dir, f'stations_in_ij_{obs_time}.txt')

    if not os.path.exists(obs_file):
        print(f'观测文件不存在：{obs_file}')
        continue

    df_obs = pd.read_csv(obs_file, sep='\t')
    if 'PM2.5' not in df_obs.columns:
        print('观测文件无 PM2.5 列')
        continue

    combined_maps = []

    # -------- Assemble all tiles --------
    for i in range(1, 8):
        for j in range(1, 12):

            tile = f'i{i}_j{j}'
            f_wrf = os.path.join(wrf_base_path, tile, f'{model_time}.npy')

            if not os.path.exists(f_wrf):
                continue

            try:
                wrf_dict = np.load(f_wrf, allow_pickle=True).item()

                pm25_base = wrf_dict['pm25'].copy()  # 完整背景场
                lat = wrf_dict['lat']
                lon = wrf_dict['lon']

                # -------- Crop region --------
                y0, y1 = 20, 100
                x0, x1 = 20, 140

                w_crop = wrf_dict['pm25'][y0:y1, x0:x1]

                def load_and_embed(path):
                    if os.path.exists(path):
                        crop = np.load(path)[10:90, 10:130]
                        pm25_base[y0:y1, x0:x1] = w_crop + crop
                    return pm25_base.copy()

                wrf1h = load_and_embed(os.path.join(wrf_1h_path, tile, f'{model_time}.npy'))
                les   = load_and_embed(os.path.join(les_base_path, tile, f'{model_time}.npy'))
                dl    = load_and_embed(os.path.join(dl_base_path, tile, f'{model_time}.npy'))

                combined_maps.append({
                    'lat': lat,
                    'lon': lon,
                    'wrf1h': wrf1h,
                    'les': les,
                    'dl': dl
                })

            except Exception as e:
                print(f'{tile} 读取失败: {e}')
                continue

    if not combined_maps:
        print('无有效模型数据')
        continue

    # -------- Interpolate to stations --------
    df_obs['wrf1h'] = interpolate_station_values(df_obs, combined_maps, 'wrf1h')
    df_obs['les']   = interpolate_station_values(df_obs, combined_maps, 'les')
    df_obs['dl']    = interpolate_station_values(df_obs, combined_maps, 'dl')

    out_csv =  f'your_path/code5FNO/Project/plot/interpolated_PM25_{model_time}.csv'
    df_obs.to_csv(out_csv, index=False)
    print(f'✅ 已保存 {out_csv}')
    all_results[model_time] = {
        'obs_time': obs_time,
        'df': df_obs
    }
print('\n全部时间完成')


In [ ]:
# Cell 2: Plot station comparison (JGR style)
import matplotlib
import matplotlib.pyplot as plt

# Set JGR compatible fonts (safe without Arial)
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 9,
    #"mathtext.fontset": "dejavusans",
    # Core: force DejaVu Sans (included by default in Matplotlib)
    "font.family": "DejaVu Sans",
    
    # Force math text to also use DejaVu Sans (critical!)
    "mathtext.fontset": "dejavusans",
    
    # Disable LaTeX rendering (avoid font conflicts)
    "text.usetex": False,
})
time_labels = ['(a)', '(b)', '(c)', '(d)']  # Support up to 4 time points
for idx, (model_time, data) in enumerate(all_results.items()):
    df = data['df']
    obs_time = data['obs_time']
    # Clean: keep only stations with all valid values > 0
    mask = (
        (df['PM2.5'] > 0) &
        (df['wrf1h'] > 0) & (df['les'] > 0) & (df['dl'] > 0) &
        df[['PM2.5', 'wrf1h', 'les', 'dl']].notna().all(axis=1)
    )
    df_clean = df[mask].copy()
    
    if len(df_clean) == 0:
        print(f"跳过绘图 {model_time}: 无有效数据")
        continue
    
    # Sort by longitude
    #df_clean = df_clean.sort_values('lat').reset_index(drop=True)
    station_id = df_clean['station_id']
   # x = np.arange(len(df_clean))
    obs = df_clean['PM2.5'].values
    wrf1h = df_clean['wrf1h'].values
    les = df_clean['les'].values
    dl = df_clean['dl'].values
    
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(station_id, obs, 'o-', color='black', label='Observation', markersize=4, linewidth=1.2)
    ax.plot(station_id, wrf1h, 's-', color='#1f77b4', label='WRF Assimilated Forecast', markersize=4, linewidth=1.2)
    ax.plot(station_id, les, '^-', color='#d62728', label='LES Forecast', markersize=4, linewidth=1.2)
    ax.plot(station_id, dl, 'D-', color='#2ca02c', label='FNO Prediction', markersize=4, linewidth=1.2)
    # === Add (a), (b) labels ===
    panel_label = time_labels[idx] if idx < len(time_labels) else f"({chr(97+idx)})"
    ax.text(0.02, 0.96, panel_label, transform=ax.transAxes,
            fontsize=14, fontweight='bold',
            verticalalignment='top', horizontalalignment='left',)
            #bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
    # Labels
    #time_label = f"{obs_time[:4]}-{obs_time[4:6]}-{obs_time[6:8]} {obs_time[8:]} UTC"
    time_label = f"{obs_time[:4]}{obs_time[4:6]}{obs_time[6:8]}{obs_time[8:]}"
    ax.set_xlabel('Station Index ', fontsize=12)
    ax.set_ylabel('PM$_{2.5}$ ($\mu$g/m$^3$)', fontsize=15)
    ax.set_title(f'PM$_{{2.5}}$ at Observation Sites ({time_label})', fontsize=15, fontweight='bold',pad=10)
    # Replace ax.legend(...)
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), 
            frameon=False, fontsize=12,)
    plt.subplots_adjust(right=0.8)  # Leave space for legend
    #ax.legend(loc='upper right', framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.5, linewidth=0.6)
    
    plt.tight_layout()
    
    # Saveyour_path/results_comparison
    #save_path = os.path.join(output_dir, f'JGR_Stations_Comparison_{model_time}.png')
    save_path = os.path.join('your_path/code5FNO/Project/plot', f'JGR_Stations_Comparison_{model_time}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✅ 已保存图: {save_path}")
    plt.show()

print("Cell 2 完成：JGR 风格站点图已生成。")

In [ ]:
# Cell 3: Error metrics summary (vs Obs and vs LES)
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def calc_metrics(y_true, y_pred, label=""):
    mask = (~np.isnan(y_true)) & (~np.isnan(y_pred)) & (y_true > 0) & (y_pred > 0)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {f'{label}_N': 0, f'{label}_MAE': np.nan, f'{label}_RMSE': np.nan, f'{label}_MRE(%)': np.nan}
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mre = np.mean(np.abs(y_pred - y_true) / y_true) * 100
    return {
        f'{label}_N': len(y_true),
        f'{label}_MAE': round(mae, 2),
        f'{label}_RMSE': round(rmse, 2),
        f'{label}_MRE(%)': round(mre, 1)
    }

summary_rows = []

for model_time, data in all_results.items():
    df = data['df']
    obs_time = data['obs_time']
    
    mask = (
        (df['PM2.5'] > 0) &
        (df['wrf1h'] > 0) & (df['les'] > 0) & (df['dl'] > 0) &
        df[['PM2.5', 'wrf1h', 'les', 'dl']].notna().all(axis=1)
    )
    df_clean = df[mask]
    
    if len(df_clean) == 0:
        continue
    
    obs = df_clean['PM2.5'].values
    wrf1h = df_clean['wrf1h'].values
    les = df_clean['les'].values
    dl = df_clean['dl'].values
    
    # 均值
    means = {
        'Obs_mean': round(np.mean(obs), 1),
        'WRF1h_mean': round(np.mean(wrf1h), 1),
        'LES_mean': round(np.mean(les), 1),
        'DL_mean': round(np.mean(dl), 1),
    }
    
    # 误差：所有模型 vs Obs
    metrics_obs = {}
    metrics_obs.update(calc_metrics(obs, wrf1h, 'WRF1h_vs_Obs'))
    metrics_obs.update(calc_metrics(obs, les, 'LES_vs_Obs'))
    metrics_obs.update(calc_metrics(obs, dl, 'DL_vs_Obs'))
    
    # 误差：DL vs LES
    metrics_dl_les = calc_metrics(les, dl, 'DL_vs_LES')
    
    row = {
        'Model_Time': model_time,
        'Obs_Time': obs_time,
        'N_valid': len(df_clean),
    }
    row.update(means)
    row.update(metrics_obs)
    row.update(metrics_dl_les)
    
    summary_rows.append(row)

# Convert to DataFrame
summary_df = pd.DataFrame(summary_rows)

# Select key columns (logically grouped)
cols_order = [
    'Model_Time', 'Obs_Time', 'N_valid',
    'Obs_mean', 'WRF1h_mean', 'LES_mean', 'DL_mean',
    'WRF1h_vs_Obs_MAE', 'WRF1h_vs_Obs_RMSE', 'WRF1h_vs_Obs_MRE(%)',
    'LES_vs_Obs_MAE', 'LES_vs_Obs_RMSE', 'LES_vs_Obs_MRE(%)',
    'DL_vs_Obs_MAE', 'DL_vs_Obs_RMSE', 'DL_vs_Obs_MRE(%)',
    'DL_vs_LES_MAE', 'DL_vs_LES_RMSE', 'DL_vs_LES_MRE(%)',
]

summary_df = summary_df[cols_order]

# Save
summary_file = os.path.join('your_path/code5FNO/Project/plot', 'Error_Metrics_Summary.csv')#output_dir
summary_df.to_csv(summary_file, index=False)
print("✅ 误差汇总表已保存至:")
print(summary_file)
print("\n误差汇总表预览:")
print(summary_df.to_string(index=False, float_format='%.2f'))

In [ ]:
# Cell: LES vs DL error heatmap
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

time_list = ['2024030105', '2024030117']
i_vals = list(range(1, 8))    # i=1..7 (列, 西→东)
j_vals = list(range(1, 12))   # j=1..11 (行, 南→北)

# JGR font settings (safe)
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 9,
    "mathtext.fontset": "dejavusans",
})
time_labels = ['(a)', '(b)', '(c)', '(d)']

for idx, target_time in enumerate(time_list):
#for target_time in time_list:
    print(f"Processing error maps for {target_time}")
    
    # Initialize error matrices (11 rows j, 7 cols i)
    mae_map = np.full((len(j_vals), len(i_vals)), np.nan)
    rmse_map = np.full_like(mae_map, np.nan)
    mre_map = np.full_like(mae_map, np.nan)  # relative error (%)
    les_mean_map = np.full_like(mae_map, np.nan)
    
    # Iterate over each tile
    for idx_i, i in enumerate(i_vals):      # i=1..7 → col 0..6
        for idx_j, j in enumerate(j_vals):  # j=1..11 → row 0..10 (南→北)
            tile_name = f'i{i}_j{j}'
            f_wrf = os.path.join(wrf_base_path, tile_name, f'{target_time}.npy')
            f_les = os.path.join(les_base_path, tile_name, f'{target_time}.npy')
            f_dl = os.path.join(dl_base_path, tile_name, f'{target_time}.npy')
            
            if not (os.path.exists(f_wrf) and os.path.exists(f_les) and os.path.exists(f_dl)):
                continue
            
            try:
                # Load data
                wrf_dict = np.load(f_wrf, allow_pickle=True).item()
                wrf_crop = wrf_dict['pm25'][20:100, 20:140]  # (80, 120)
                
                les_crop = np.load(f_les, allow_pickle=True)[10:90, 10:130]  # (80, 120)
                dl_crop = np.load(f_dl, allow_pickle=True)[10:90, 10:130]    # (80, 120)
                
                # Build complete field
                les_full = wrf_crop + les_crop
                dl_full = wrf_crop + dl_crop
                
                # Only consider valid values (>0)
                valid = (les_full > 0) & (dl_full > 0)
                if not np.any(valid):
                    continue
                
                les_valid = les_full[valid]
                dl_valid = dl_full[valid]
                
                # Calculate metrics
                mae = mean_absolute_error(les_valid, dl_valid)
                rmse = np.sqrt(mean_squared_error(les_valid, dl_valid))
                les_mean = np.mean(les_valid)
                mre = mae / les_mean * 100  # Percentage
                
                # Store (note: j_vals south to north, but heatmap top to bottom = north to south)
                # So we reverse j index: row = (10 - idx_j)
                row = len(j_vals) - 1 - idx_j  # j=11 → row=0 (top), j=1 → row=10 (bottom)
                col = idx_i                    # i=1 → col=0 (left), i=7 → col=6 (right)
                
                mae_map[row, col] = mae
                rmse_map[row, col] = rmse
                mre_map[row, col] = mre
                les_mean_map[row, col] = les_mean
                
            except Exception as e:
                print(f"  Error in {tile_name}: {e}")
                continue
    
    # --- Draw three heatmaps ---
    titles = ['MAE (μg/m$^3$)', 'RMSE (μg/m$^3$)', 'MRE (%)']
    data_maps = [mae_map, rmse_map, mre_map]
    
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    
    for ax, data, title in zip(axes, data_maps, titles):
        # Heatmap (no geographic projection, pure matrix)
        if title=='MRE (%)':
            im = ax.imshow(data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=30)
        else:
            im = ax.imshow(data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=10)
        
        # Set axes
        ax.set_xticks(np.arange(len(i_vals)))
        ax.set_xticklabels([f'i{i}' for i in i_vals], rotation=0)
        ax.set_yticks(np.arange(len(j_vals)))
        ax.set_yticklabels([f'j{j}' for j in reversed(j_vals)])  # j=11 at top
        
        # Add value labels
        for i in range(len(i_vals)):
            for j in range(len(j_vals)):
                val = data[j, i]
                if not np.isnan(val):
                    text_color ='black'# 'white' if val > np.nanpercentile(data, 70) else 'black'
                    ax.text(i, j, f'{val:.1f}', ha='center', va='center', 
                            color=text_color, fontsize=10, weight='bold')
        
        ax.set_title(title, fontsize=15,fontweight='bold',)
        plt.colorbar(im, ax=ax, shrink=0.8, pad=0.05)
    # === Add (a), (b) labels ===
    if idx==0:
        axes[0].text(-0., 1.08, '(a)', transform=axes[0].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
        axes[1].text(-0., 1.08, '(b)', transform=axes[1].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
        axes[2].text(-0., 1.08, '(c)', transform=axes[2].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
    else:
        axes[0].text(-0., 1.08, '(d)', transform=axes[0].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
        axes[1].text(-0., 1.08, '(e)', transform=axes[1].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
        axes[2].text(-0., 1.08, '(f)', transform=axes[2].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')

    plt.tight_layout()
    out_file = os.path.join(out_dir, f'LES_DL_Error_Heatmaps_{target_time}.png')
    plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✅ Saved: {out_file}")
    plt.show()
    plt.close(fig)
        # --- 计算全局平均误差（所有有效 tile 的平均）---
    valid_mae = mae_map[~np.isnan(mae_map)]
    valid_rmse = rmse_map[~np.isnan(rmse_map)]
    valid_mre = mre_map[~np.isnan(mre_map)]
    
    if len(valid_mae) > 0:
        global_mae = np.mean(valid_mae)
        global_rmse = np.mean(valid_rmse)
        global_mre = np.mean(valid_mre)
        
        print(f"\n=== 全局平均误差 (LES vs FNO, {target_time}) ===")
        print(f"平均 MAE: {global_mae:.3f} μg/m³")
        print(f"平均 RMSE: {global_rmse:.3f} μg/m³")
        print(f"平均 MRE: {global_mre:.3f}%")
        print(f"有效 tile 数: {len(valid_mae)} / {len(i_vals)*len(j_vals)}\n")
        
    else:
        print(f"  无有效数据用于全局统计 ({target_time})")
print("All error heatmaps completed.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- JGR 字体设置 ---
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "mathtext.fontset": "dejavusans",
    "font.family": "DejaVu Sans",
})

# --- 指定 cases (time, i, j) ---
cases = [
    #('2024030105', 4, 1),
    ('2024030105', 6, 7),
    #('2024030105', 7, 8),
    ('2024030105', 5, 10),
    #('2024030117', 7, 5),
    ('2024030117', 1, 4),
]

n_cases = len(cases)

# --- 预加载所有数据 ---
les_total_list = []
dl_total_list = []
title_labels = []
error_info_list = []

vmin_total, vmax_total = 0, 60  # total PM2.5 范围

for time_str, i, j in cases:
    tile_name = f'i{i}_j{j}'
    try:
        # 加载 WRF
        wrf_file = os.path.join(wrf_base_path, tile_name, f'{time_str}.npy')
        wrf_dict = np.load(wrf_file, allow_pickle=True).item()
        wrf_crop = wrf_dict['pm25'][20:100, 20:140]  # 80x120
        
        # 加载 LES
        les_file = os.path.join(les_base_path, tile_name, f'{time_str}.npy')
        les_delta = np.load(les_file, allow_pickle=True)
        les_crop = les_delta[10:90, 10:130] if les_delta.shape == (100, 140) else les_delta
        
        # 加载 FNO
        dl_file = os.path.join(dl_base_path, tile_name, f'{time_str}.npy')
        dl_delta = np.load(dl_file, allow_pickle=True)
        dl_crop = dl_delta[10:90, 10:130] if dl_delta.shape == (100, 140) else dl_delta
        
        # 构建 total
        les_total = wrf_crop + les_crop
        dl_total = wrf_crop + dl_crop
        
        # 计算误差
        valid = (les_total > 0) & (dl_total > 0)
        if np.any(valid):
            mae = mean_absolute_error(les_total[valid], dl_total[valid])
            rmse = np.sqrt(mean_squared_error(les_total[valid], dl_total[valid]))
            mre = mae / np.mean(les_total[valid]) * 100
        else:
            mae, rmse, mre = np.nan, np.nan, np.nan
        
        les_total_list.append(les_total)
        dl_total_list.append(dl_total)
        title_labels.append(f'{int(time_str)+1} i{i}j{j}')
        error_info_list.append((mae, rmse, mre))
        
    except Exception as e:
        print(f"跳过 {tile_name} @ {time_str}: {e}")
        les_total_list.append(None)
        dl_total_list.append(None)
        title_labels.append(f'i{i}j{j} (missing)')
        error_info_list.append((np.nan, np.nan, np.nan))

# --- 绘图：2行 × n_cases 列 ---
fig, axes = plt.subplots(2, n_cases, figsize=(4 * n_cases+3, 5))

# 如果只有一列，确保 axes 是 2D
if n_cases == 1:
    axes = axes.reshape(2, 1)

# 上行：LES
for col in range(n_cases):
    if les_total_list[col] is not None:
        im1 = axes[0, col].imshow(
            les_total_list[col], 
            cmap='jet', 
            vmin=vmin_total, 
            vmax=vmax_total, 
            origin='lower'
        )
        axes[0, col].set_title(title_labels[col], fontsize=16,)#fontweight='bold',
    else:
        axes[0, col].text(0.5, 0.5, 'Missing', ha='center', va='center')
    axes[0, col].axis('off')

    if col==0:
        axes[0,0].text(0.02, 0.97, '(a)', transform=axes[0,0].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
    if col==1:
        axes[0,1].text(0.02, 0.97, '(b)', transform=axes[0,1].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
    if col==2:
        axes[0,2].text(0.02, 0.97, '(c)', transform=axes[0,2].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')

# 下行：FNO
for col in range(n_cases):
    if dl_total_list[col] is not None:
        im2 = axes[1, col].imshow(
            dl_total_list[col], 
            cmap='jet',#'YlOrRd', 
            vmin=vmin_total, 
            vmax=vmax_total, 
            origin='lower'
        )
        # 在图下方添加误差信息
        mae, rmse, mre = error_info_list[col]
        if not np.isnan(mae):
            error_text = f'MAE={mae:.1f}'
            axes[1, col].text(0.5, -0.05, error_text, ha='center', va='top',
                              transform=axes[1, col].transAxes, fontsize=16,)#,fontweight='bold'
    else:
        axes[1, col].text(0.5, 0.5, 'Missing', ha='center', va='center')
    axes[1, col].axis('off')
    if col==0:
        axes[1,0].text(0.02, 0.97, '(d)', transform=axes[1,0].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
    if col==1:
        axes[1,1].text(0.02, 0.97, '(e)', transform=axes[1,1].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')
    if col==2:
        axes[1,2].text(0.02, 0.97, '(f)', transform=axes[1,2].transAxes,
                 fontsize=14,  va='top', ha='left',fontweight='bold')

# 共用 colorbar（放在右侧）
cbar = fig.colorbar(im1, ax=axes, shrink=0.7, aspect=30, pad=0.02)
cbar.set_label('PM$_{2.5}$ ($\mu$g/m$^3$)', fontsize=14)

# 行标签
fig.text(0.1, 0.705, 'LES Forecast', rotation=90, fontsize=16, va='center')#fontweight='bold',
fig.text(0.1, 0.29, 'FNO Prediction', rotation=90, fontsize=16, va='center')
#plt.tight_layout(rect=[0.06, 0.02, 0.92, 1])  # [left, bottom, right, top]
#plt.tight_layout(rect=[0.05, 0, 1, 1])  # 为行标签留空间
out_file = os.path.join('your_path/code5FNO/Project/plot', f'detial_ij.png')
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import fftpack, interpolate
import os
import matplotlib.ticker as ticker

# ================= 1. 路径与配置 =================
wrf1h_base_path = 'your_path/d03_d06_pm25'

wrf_u_path = 'your_path/120_160/all_data/U10'
wrf_v_path = 'your_path/120_160/all_data/V10'
les_u_path = 'your_path/data/LES_U10'
les_v_path = 'your_path/data/LES_V10'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

time_list = ['2024030105', '2024030117']
labels = ['(a) Daytime', '(b) Nighttime']

# 物理参数
DX = 100.0        
Z_HEIGHT = 12.0   
LIMIT_WAVELENGTH_KM = 0.6 

# 切片
sl_pm25_h = slice(28, 92) 
sl_pm25_w = slice(48, 112)
sl_res_h = slice(18, 82)
sl_res_w = slice(38, 102)
sl_wrf_wind_h = slice(18, 82)
sl_wrf_wind_w = slice(38, 102)
sl_les_wind_h = slice(28, 92)
sl_les_wind_w = slice(48, 112)

plt.rcParams.update({
    "font.size": 14,
    "mathtext.fontset": "dejavusans",
    "font.family": "DejaVu Sans",
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.labelsize": 15,
})

# ================= 2. 核心计算函数 =================
def get_norm_freq_spectrum(image, u_mean, dx, z_height):
    if u_mean < 0.1: return None, None
    image = np.nan_to_num(image, nan=0.0)
    variance = np.var(image)
    if variance == 0: return None, None
    image = image - np.mean(image)
    
    h, w = image.shape
    win = np.outer(np.hanning(h), np.hanning(w))
    image_windowed = image * win
    correction = 1.0 / np.mean(win**2)
    
    F = fftpack.fftshift(fftpack.fft2(image_windowed))
    psd2D = np.abs(F)**2 * correction
    
    center = (h//2, w//2)
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)
    max_r = min(h, w) // 2 
    
    tbin = np.bincount(r.ravel(), weights=psd2D.ravel())
    if len(tbin) > max_r: tbin = tbin[:max_r]
    
    L_total = w * dx 
    k_indices = np.arange(len(tbin))
    
    n_Hz = k_indices * u_mean / L_total
    f_dimensionless = n_Hz * z_height / u_mean
    
    norm_power = tbin / variance
    return f_dimensionless[1:], norm_power[1:]

def f_to_wavelength_km(f):
    return Z_HEIGHT / (f * 1000.0)

def wavelength_km_to_f(lam_km):
    return Z_HEIGHT / (lam_km * 1000.0)

F_CUTOFF = wavelength_km_to_f(LIMIT_WAVELENGTH_KM)

# ================= 3. 统计函数 =================
def calculate_specific_stats(f_axis, psd_wrf, psd_les, psd_fno, f_cutoff):
    mask_left = (f_axis <= f_cutoff)
    mask_right = (f_axis > f_cutoff)
    
    def safe_integ(y, x, mask):
        m = mask & ~np.isnan(y) & ~np.isnan(x)
        if np.sum(m) < 2: return 0.0
        return np.trapz(y[m], x[m])

    # 1. 基准：LES 左侧
    e_les_left = safe_integ(psd_les, f_axis, mask_left)
    if e_les_left == 0: return 0, 0, 0

    # 2. 左侧统计
    e_wrf_left = safe_integ(psd_wrf, f_axis, mask_left)
    e_fno_left = safe_integ(psd_fno, f_axis, mask_left)
    
    # 3. 右侧统计 (仅 LES)
    e_les_right = safe_integ(psd_les, f_axis, mask_right)
    
    # 4. 百分比
    r_wrf_left_ratio = (e_wrf_left / e_les_left) * 100
    r_fno_left_ratio = (e_fno_left / e_les_left) * 100
    r_les_right_ratio = (e_les_right / e_les_left) * 100
    
    return r_wrf_left_ratio, r_fno_left_ratio, r_les_right_ratio

# ================= 4. 主程序 =================
fig, axes = plt.subplots(1, 2, figsize=(16, 7.8)) 
common_f = np.logspace(-3.1, -0.9, 200) 

for idx, target_time in enumerate(time_list):
    ax = axes[idx]
    print(f"Processing {target_time} ...")
    
    psd_list_wrf, psd_list_les, psd_list_dl = [], [], []
    
    for i in range(1, 8):
        for j in range(1, 12):
            tile = f'i{i}_j{j}'
            p_wrf_base = os.path.join(wrf_base_path, tile, f'{target_time}.npy')
            p_wrf_1h   = os.path.join(wrf1h_base_path, tile, f'{target_time}.npy')
            p_les      = os.path.join(les_base_path, tile, f'{target_time}.npy')
            p_dl       = os.path.join(dl_base_path, tile, f'{target_time}.npy')
            p_wu       = os.path.join(wrf_u_path, tile, f'{target_time}.npy')
            p_wv       = os.path.join(wrf_v_path, tile, f'{target_time}.npy')
            p_lu       = os.path.join(les_u_path, tile, f'{target_time}.npy')
            p_lv       = os.path.join(les_v_path, tile, f'{target_time}.npy')
            
            check = [p_wrf_base, p_wrf_1h, p_les, p_dl, p_wu, p_wv, p_lu, p_lv]
            if all(os.path.exists(p) for p in check):
                try:
                    base = np.load(p_wrf_base, allow_pickle=True).item()['pm25'][sl_pm25_h, sl_pm25_w]
                    val_wrf = base + np.load(p_wrf_1h, allow_pickle=True)[sl_res_h, sl_res_w]
                    val_les = base + np.load(p_les, allow_pickle=True)[sl_res_h, sl_res_w]
                    val_dl  = base + np.load(p_dl, allow_pickle=True)[sl_res_h, sl_res_w]
                    wu = np.load(p_wu, allow_pickle=True)[sl_wrf_wind_h, sl_wrf_wind_w]
                    wv = np.load(p_wv, allow_pickle=True)[sl_wrf_wind_h, sl_wrf_wind_w]
                    lu = np.load(p_lu, allow_pickle=True)
                    lv = np.load(p_lv, allow_pickle=True)
                    if lu.shape[0] > 64: 
                        lu = lu[sl_les_wind_h, sl_les_wind_w]
                        lv = lv[sl_les_wind_h, sl_les_wind_w]
                    
                    if val_wrf.shape != (64, 64): continue
                    spd_wrf = np.mean(np.sqrt(wu**2 + wv**2))
                    spd_les = np.mean(np.sqrt(lu**2 + lv**2))
                    spd_dl  = spd_les
                    
                    f_w, p_w = get_norm_freq_spectrum(val_wrf, spd_wrf, DX, Z_HEIGHT)
                    f_l, p_l = get_norm_freq_spectrum(val_les, spd_les, DX, Z_HEIGHT)
                    f_d, p_d = get_norm_freq_spectrum(val_dl,  spd_dl,  DX, Z_HEIGHT)
                    
                    if f_w is None or f_l is None: continue
                    interp_w = np.interp(common_f, f_w, p_w, left=np.nan, right=np.nan)
                    interp_l = np.interp(common_f, f_l, p_l, left=np.nan, right=np.nan)
                    interp_d = np.interp(common_f, f_d, p_d, left=np.nan, right=np.nan)
                    psd_list_wrf.append(interp_w); psd_list_les.append(interp_l); psd_list_dl.append(interp_d)
                except Exception: continue

    if not psd_list_wrf: continue
    avg_wrf = np.nanmean(psd_list_wrf, axis=0)
    avg_les = np.nanmean(psd_list_les, axis=0)
    avg_dl  = np.nanmean(psd_list_dl,  axis=0)
    
    # 统计
    r_wrf_L, r_fno_L, r_les_R = calculate_specific_stats(
        common_f, avg_wrf, avg_les, avg_dl, F_CUTOFF
    )
    
    cut_idx = np.searchsorted(common_f, F_CUTOFF)
    f_valid = common_f[:cut_idx+1]
    f_grey  = common_f[cut_idx:]
    
    # 1. 绘图 (按照频率绘制)
    # WRF
    ax.loglog(f_valid, avg_wrf[:cut_idx+1], 'b--', label='WRF Assimilated', linewidth=2, alpha=0.7)
    ax.loglog(f_grey,  avg_wrf[cut_idx:],   color='lightgray', linestyle='--', linewidth=2, alpha=0.6) 
    # LES (Legend 改为 LES Forecast, 去掉 Truth)
    ax.loglog(f_valid, avg_les[:cut_idx+1], 'r-',  label='LES Forecast',    linewidth=2.5, alpha=0.9)
    ax.loglog(f_grey,  avg_les[cut_idx:],   color='lightgray', linestyle='-',  linewidth=2.5, alpha=0.6)
    # FNO
    ax.loglog(f_valid, avg_dl[:cut_idx+1],  'g-.', label='FNO Prediction',  linewidth=2.5, alpha=0.8)
    ax.loglog(f_grey,  avg_dl[cut_idx:],    color='lightgray', linestyle='-.', linewidth=2.5, alpha=0.6)

    # 参考线
    f_ref = np.array([2e-3, 1e-2])  
    psd_ref = f_ref**(-5/3)         
    scale_factor = 1000.  
    ax.loglog(f_ref, scale_factor * psd_ref, 'k:', linewidth=2, label=r'$f^{-5/3}$')
    
    # 垂直线
    ax.axvline(x=F_CUTOFF, color='gray', linestyle=':', linewidth=1.5)
    ax.text(F_CUTOFF*1.1, 1e2, r'$\lambda=0.6$km', color='gray', rotation=90, fontsize=14)

    # === 精简版文本框 ===
    # 将描述性文字极度精简，但保留核心信息
    stats_text = (
        f"vs. LES Effective TKE:\n"
        f"----------------------\n"
        f"Effective (>0.6km):\n"
        f"  WRF: {r_wrf_L:.1f}%\n"
        f"  FNO: {r_fno_L:.1f}%\n"
        f"Spurious (<0.6km):\n"
        f"  LES: {r_les_R:.1f}%"
    )
    
    props = dict(boxstyle='round', facecolor='white', alpha=0.92, edgecolor='darkgray')
    ax.text(0.04, 0.04, stats_text, transform=ax.transAxes, fontsize=14,
            verticalalignment='bottom', bbox=props, family='monospace')

    # === 关键修改：坐标轴位置交换 ===
    
    # 1. 设置主轴（当前是 Frequency）到顶部
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top') 
    ax.set_xlabel('Normalized Frequency $f = n z / U$', fontsize=16, labelpad=10)
    
    # 2. 添加次坐标轴（Wavelength）到底部
    # 注意：Wavelength 变换函数 f = z/lambda
    secax = ax.secondary_xaxis('bottom', functions=(f_to_wavelength_km, wavelength_km_to_f))
    secax.set_xlabel('Wavelength $\lambda$ (km)', fontsize=16)

    ax.set_ylabel('Normalized PSD $S(n)/\sigma^2$', fontsize=16)
    ax.set_xlim(1e-3, 0.2) 
    
    ax.set_title(f'{labels[idx]} Spectrum ({int(target_time)+1})', fontsize=16, fontweight='bold', pad=20)
    ax.legend(fontsize=14, loc='upper right')
    ax.grid(True, which="both", alpha=0.2)

plt.tight_layout()
out_file = os.path.join(output_dir, 'spectrum_final_swapped_axes.png')
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved to {out_file}")
plt.show()

### fea_important

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from datetime import datetime
import segmentation_models_pytorch as smp
import sys
sys.path.append('your_path/code5FNO/Project/code')
import joblib
from data import DataReader,NumpyDataset,HybridNormalizer
from sklearn.preprocessing import RobustScaler

time_pairs = [('2024030105', '2024030106'),('2024030117', '2024030118'),
              ]
# Directory configuration dictionary
directories = {
    'pm25': 'your_path/d03_d06_pm25',
    'output': 'your_path/moni_pm25',
    'T2': 'your_path/120_160/all_data/T2',
    'E_pm25': 'your_path/120_160/all_data/E_pm25',
    'HGT': 'your_path/120_160/all_data/HGT',
    'U10': 'your_path/120_160/all_data/U10',
    'V10': 'your_path/120_160/all_data/V10',
    'LAI': 'your_path/120_160/all_data/LAI',
    'QVAPOR': 'your_path/120_160/all_data/QVAPOR',
    'PBLH':'your_path/120_160/all_data/PBLH',
    'UST':'your_path/120_160/all_data/UST',
    'HFX':'your_path/120_160/all_data/HFX',
}
val_data_reader = DataReader(i_range=range(1, 8), j_range=range(1, 12), 
                directories=directories, 
    time_pairs=time_pairs,
)

val_normalizer = HybridNormalizer()
val_normalizer.load("your_path/code5FNO/Project/code/hybrid_normalizer.pkl")  # Load the hybrid normalizer saved during training
# 3. Create validation dataset
val_dataset = NumpyDataset(
    data_reader=val_data_reader,
    normalizer=val_normalizer,  # Use training set normalizer
    transform=None,  
)
# 4. Create validation dataloader
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import copy
import os
from FNO import FNO2d
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Output folder path
output_directory = 'your_path/code5FNO/Project/plot'

# Define time list (uncomment or add as needed)
time_pairs = [('2024030105', '2024030106'),('2024030117', '2024030118'),
]

# Load model
in_channels_count = len(directories) - 1
model=FNO2d(in_channels=len(directories)-1, out_channels=1, 
              modes1=20, modes2=24, width=32).to(device)
checkpoint = torch.load('your_path/code5FNO/Project/code/best_fno_model.pth', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load normalizer
val_normalizer = HybridNormalizer()
val_normalizer.load("your_path/code5FNO/Project/code/hybrid_normalizer.pkl")

import pandas as pd
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def calculate_real_world_importance_no_seaborn(model, dataloader, normalizer, feature_names, device):
    """
    计算基于反归一化（真实物理量）的置换特征重要性，并仅在中心 80x120 区域计算误差
    """
    model.eval()
    
    # === 裁剪参数 ===
    H_START, H_END = 10, 90   # 100 -> 80
    W_START, W_END = 10, 130  # 140 -> 120

    # 1. 准备数据（加载全部）
    print("正在加载全部验证集数据以进行特征分析...")
    all_inputs = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets, _, _ in dataloader:
            all_inputs.append(inputs)
            all_targets.append(targets)
    
    X_full = torch.cat(all_inputs, dim=0).to(device)
    y_full = torch.cat(all_targets, dim=0).to(device)
    
    # 2. 定义裁剪 + 反归一化 + MAE 函数
    def calculate_mae_on_crop(preds_tensor, truth_tensor):
        # 裁剪中心 80x120
        preds_crop = preds_tensor[:, :, H_START:H_END, W_START:W_END]
        truth_crop = truth_tensor[:, :, H_START:H_END, W_START:W_END]
        
        # Denormalize
        preds_np = preds_crop.squeeze(1).cpu().numpy()
        truth_np = truth_crop.squeeze(1).cpu().numpy()
        
        preds_real_list = []
        truth_real_list = []
        for i in range(len(preds_np)):
            preds_real_list.append(normalizer.inverse_transform(preds_np[i], 'output'))
            truth_real_list.append(normalizer.inverse_transform(truth_np[i], 'output'))
        
        preds_real = np.array(preds_real_list)
        truth_real = np.array(truth_real_list)
        
        # 计算 MAE
        return np.mean(np.abs(preds_real - truth_real))

    # 3. 计算基础误差 (Baseline MAE)
    print("正在计算基础误差 (Baseline)...")
    with torch.no_grad():
        baseline_preds = model(X_full)
        baseline_mae = calculate_mae_on_crop(baseline_preds, y_full)
    
    print(f"基础 MAE (真实物理量, 80x120 区域): {baseline_mae:.4f} μg/m³")

    # 4. 特征重要性分析
    input_features = [k for k in feature_names.keys() if k != 'output']
    results = []
    
    print(f"开始分析 {len(input_features)} 个特征...")
    
    for idx, name in enumerate(input_features):
        # 置换该特征
        X_permuted = X_full.clone()
        perm_indices = torch.randperm(X_permuted.size(0))
        X_permuted[:, idx, :, :] = X_full[perm_indices, idx, :, :]
        
        # 前向传播 + 裁剪 + 计算 MAE
        with torch.no_grad():
            perm_preds = model(X_permuted)
            perm_mae = calculate_mae_on_crop(perm_preds, y_full)
        
        importance_delta = perm_mae - baseline_mae
        
        results.append({
            'Feature': name,
            'Permuted_MAE': perm_mae,
            'Importance_Delta': importance_delta,
            'Relative_Change_%': (importance_delta / baseline_mae) * 100
        })
        print(f" - {name}: MAE={perm_mae:.4f} (+{importance_delta:.4f})")

    # Convert to DataFrame
    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(by='Importance_Delta', ascending=True).reset_index(drop=True)
    
    return df_results, baseline_mae

# ================= 运行分析 =================

# 运行函数
df_importance, base_mae = calculate_real_world_importance_no_seaborn(
    model, val_dataloader, val_normalizer, directories, device
)

# ================= 打印表格 =================
print("\n========== 特征重要性排序 (基于反归一化 Loss) ==========")
# 重新按降序排序以便打印查看
print_df = df_importance.sort_values(by='Importance_Delta', ascending=False)
print(print_df[['Feature', 'Importance_Delta', 'Permuted_MAE']])

# ================= 绘图 (纯 Matplotlib) =================
plt.figure(figsize=(10, 8))

# 生成颜色：根据数值大小生成渐变色
# 使用 'coolwarm' 色谱，数值越大越红
norm = mcolors.Normalize(vmin=df_importance['Importance_Delta'].min(), vmax=df_importance['Importance_Delta'].max())
cmap = cm.get_cmap('coolwarm') 
colors = [cmap(norm(val)) for val in df_importance['Importance_Delta']]

# 绘制横向条形图 (barh)
bars = plt.barh(df_importance['Feature'], df_importance['Importance_Delta'], color=colors, edgecolor='grey')

# 添加网格线
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Add value labels
for bar in bars:
    width = bar.get_width()
    # Labels位置稍微往右偏一点
    label_x_pos = width if width > 0 else 0
    plt.text(label_x_pos, bar.get_y() + bar.get_height()/2, 
             f' +{width:.3f}', 
             va='center', ha='left', fontsize=10, fontweight='bold')
plt.title(f'Permutation Feature Importance (Real-world PM2.5 Error)\nBaseline MAE: {base_mae:.2f} $\mu g/m^3$', fontsize=16, pad=15)
plt.xlabel('Increase in MAE ($\mu g/m^3$) when feature is shuffled', fontsize=15)
plt.ylabel('Input Variables', fontsize=15)
plt.tight_layout()
# Save image
save_path = 'your_path/code5FNO/Project/plot/feature_importance_mpl.png'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt.savefig(save_path, dpi=300, bbox_inches='tight')
#print(f"图表已保存至: {save_path}")
plt.show()